# 📊 Final Model Evaluation & Results Analysis

Once the model is trained, we need to analyze why it makes certain classification errors. This notebook helps visualize the **Confusion Matrix** and **Precision/Recall** for each command.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import os
import sys

sys.path.append('..')
import config
from model import EEGClassifier

## 1. Load Data
We use the test set (`X_test.npy`) which was not seen by the model during training or validation.

In [ ]:
X_test = np.load(os.path.join('..', config.DATA_DIR, 'X_test.npy'))
y_test = np.load(os.path.join('..', config.DATA_DIR, 'y_test.npy'))

print(f"📌 Test Data Shape: {X_test.shape}")

## 2. Load Best Model
Importing the saved weights from `models/best_model.pth`.

In [ ]:
model = EEGClassifier(use_cnn_lstm=config.USE_CNN_LSTM).to(config.DEVICE)
model_path = os.path.join('..', config.BEST_MODEL_PATH)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=config.DEVICE))
    model.eval()
    print("✅ Best model loaded successfully.")
else:
    print(f"❌ ERROR: {model_path} not found! Run train.py first.")

## 3. Generate Predictions
Let's predict labels for our unseen test set.

In [ ]:
X_test_tensor = torch.from_numpy(X_test).float().to(config.DEVICE)

with torch.no_grad():
    outputs = model(X_test_tensor)
    _, y_pred = torch.max(outputs.data, 1)

y_pred = y_pred.cpu().numpy()
y_true = y_test

## 4. Confusion Matrix
This matrix reveals if the model is confusing **LEFT** blinks with **RIGHT** blinks, or if **FORWARD** (focus) is mixing with **IDLE** (relaxing).

In [ ]:
cm = confusion_matrix(y_true, y_pred)
labels = [config.CLASSES[i] for i in range(len(config.CLASSES))]

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title("🧠 Confusion Matrix: Predicted vs True")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

## 5. Classification Report
Look for **F1-Score** — it's the best measure when classes might be slightly unbalanced.

In [ ]:
print("📋 Classification Report")
print(classification_report(y_true, y_pred, target_names=labels))

## 6. Confidence Analysis
Does our model have a high confidence score on the test set?

In [ ]:
import torch.nn.functional as F
probs = F.softmax(outputs, dim=1)
confidences, _ = torch.max(probs, 1)
confidences = confidences.cpu().numpy()

plt.figure(figsize=(10, 5))
plt.hist(confidences, bins=20, color='skyblue', edgecolor='black')
plt.title("Distribution of Prediction Confidence")
plt.xlabel("Confidence Score (0-1)")
plt.ylabel("Count")
plt.axvline(x=config.CONFIDENCE_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({config.CONFIDENCE_THRESHOLD})')
plt.legend()
plt.show()